# 01. 2024 원본으로 2026년 하루치 시뮬레이션 데이터 만들기

| 항목 | 내용 |
| --- | --- |
| 입력 | `pdm-project/data/processed/cal01_base_v1.csv` (2024년 1초 원기록, 22채널, 제외 표시 포함) |
| 출력 | `data/sim/cal01_sim_2026-10-26.csv` (1초 86,400행, 서버 등록 형식), `data/sim/cal01_sim_2026-10-26_events.csv` (심은 사건 정답) |
| 이 노트북이 하는 일 | 2024년 LOT의 실제 구간을 2026년 하루 시간축으로 옮겨 붙이고, 작은 흔들림을 더한 뒤, 확인된 이상 사건 모양을 심음 |
| 서버 등록 | `.venv/Scripts/python.exe backend/manage.py import_cal01 data/sim/cal01_sim_2026-10-26.csv --name "2026 시뮬레이션"` |

**왜 VAE가 아니라 구간 붙이기(블록 샘플링)인가**

- LOT이 14개뿐이라 VAE 같은 생성 모델이 22채널의 시간 구조와 채널 사이 관계를 배우고 검증하기에는 자료가 적음
- 실제 구간을 통째로 옮기면 공통 움직임(Z3 봉우리가 Z4~Z6로 번짐), TC와 OP의 반대 움직임 같은 관계가 그대로 보존됨
- 어느 2024 구간에서 왔는지 기록이 남아 설명할 수 있음. 시연 화면에 "시뮬레이션 자료"라고 밝히는 것이 전제임

**만드는 규칙**

| 규칙 | 값 |
| --- | --- |
| 날짜 | 2026-10-26 00:00:00 ~ 23:59:59, 1초 간격 |
| LOT 일정 | 4개 LOT (SPHC, SPCE, SPFH590, SPHC). 각 LOT은 같은 강종의 2024 LOT에서 제외 구간이 없는 이어진 행을 무작위 시작점으로 뽑음 |
| 흔들림 | 채널마다 그 구간 표준편차의 10% 크기 정규 잡음. OP는 0~100 안으로, CP-Z4M은 그대로 |
| 심는 사건 | 이식 2건 (팀 확정 사건 T01, T12의 CP-Z4 편차 모양), 합성 1건 (가열 능력 저하: OP-Z4가 오르는데 TC-Z4가 내려감) |
| 정답 표시 | 심은 사건 행과 옮겨 온 구간에 있던 팀 판단 1 행은 `LABEL = 1`, 나머지 0. `LABEL_VERSION = sim-2026-v1`, `LABEL_REASON`에 사건 번호 또는 원본 T 번호 |
| 재현 | `RANDOM_STATE = 42` 고정 |

- 원본 시각을 2026년으로 바꾸는 것은 이 시뮬레이션 파일 안에서만임. 2024 원본 자료의 시각은 건드리지 않음
- 이 파일은 `data/` 아래라 Git에 올라가지 않음. 노트북만 올림


## 1. 경로와 설정

In [ ]:
import os
import pandas as pd
import numpy as np

# 저장소 루트 찾기: backend 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "backend")):
        ROOT = os.path.dirname(ROOT)

# 2024 원본은 옆 저장소 pdm-project에 있음
PDM_DIR = os.path.join(os.path.dirname(ROOT), "pdm-project")
SOURCE_CSV = os.path.join(PDM_DIR, "data", "processed", "cal01_base_v1.csv")
ENC = "utf-8-sig"

# 만들 파일 (data 폴더는 Git에 올라가지 않음)
SIM_DATE = "2026-10-26"
OUT_DIR = os.path.join(ROOT, "data", "sim")
OUT_CSV = os.path.join(OUT_DIR, "cal01_sim_" + SIM_DATE + ".csv")
EVENT_CSV = os.path.join(OUT_DIR, "cal01_sim_" + SIM_DATE + "_events.csv")
FIG_PNG = os.path.join(OUT_DIR, "cal01_sim_" + SIM_DATE + ".png")
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42   # 프로젝트 가이드대로 고정
JITTER = 0.1        # 흔들림 크기: 채널 표준편차의 10%
LABEL_VERSION = "sim-2026-v1"

# 서버가 요구하는 22채널 (backend/monitor/channels.py와 같은 순서)
CHANNELS = ["CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2", "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6",
            "OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6", "CP-Z4", "CP-Z4M", "TC-Z7", "TC-Z8", "TC-Z9",
            "OP-Z7", "OP-Z8", "TC-OUT1", "TC-OUT2"]

# 하루 LOT 일정 (합계 1,440분). 원본 LOT은 같은 강종이고 제외 구간 없이 이어진 행이 길이보다 길어야 함
SCHEDULE = [
    {"LOT_NO": "260001", "GRADE": "SPHC",    "SOURCE_LOT": 240069, "START": "00:00", "MINUTES": 390},
    {"LOT_NO": "260002", "GRADE": "SPCE",    "SOURCE_LOT": 240121, "START": "06:30", "MINUTES": 360},
    {"LOT_NO": "260003", "GRADE": "SPFH590", "SOURCE_LOT": 240108, "START": "12:30", "MINUTES": 330},
    {"LOT_NO": "260004", "GRADE": "SPHC",    "SOURCE_LOT": 240091, "START": "18:00", "MINUTES": 360},
]

# 심을 사건. 이식은 2024 확정 사건의 CP-Z4 편차 모양을 그대로 옮김, 합성은 식으로 만듦
EVENTS = [
    {"EVENT_ID": "E1", "KIND": "이식", "TITLE": "CP-Z4 급등 후 복귀 (T01 모양)",
     "TARGET_LOT": "260002", "START": "09:10",
     "SOURCE_LOT": 240069, "SOURCE_START": "2024-07-08 02:28:00", "SOURCE_END": "2024-07-08 02:41:59"},
    {"EVENT_ID": "E2", "KIND": "이식", "TITLE": "CP-Z4 선행 하강, 정지 직전 (T12 모양)",
     "TARGET_LOT": "260004", "START": "21:30",
     "SOURCE_LOT": 240058, "SOURCE_START": "2024-06-09 07:32:00", "SOURCE_END": "2024-06-09 07:59:59"},
    {"EVENT_ID": "E3", "KIND": "합성", "TITLE": "가열 능력 저하: OP-Z4 상승에도 TC-Z4 하강",
     "TARGET_LOT": "260003", "START": "15:00", "RAMP_MINUTES": 40, "RECOVER_MINUTES": 10,
     "TC_DROP": 6.0, "OP_RISE": 12.0},
]

## 2. 2024 원본 읽기 (제외 구간은 뺌)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED"] + CHANNELS
source = pd.read_csv(SOURCE_CSV, encoding=ENC, usecols=columns)
source["MEAS_DT"] = pd.to_datetime(source["MEAS_DT"], errors="raise")
source = source.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

# 정지, 승온처럼 팀이 제외한 구간은 쓰지 않음
usable = source[source["EXCLUDED"] == False].reset_index(drop=True)

print("원본 행:", len(source), "/ 제외 뺀 행:", len(usable))
usable.groupby("LOT_NO").agg(강종=("GRADE", "first"), 행=("MEAS_DT", "count"))

## 3. LOT마다 2024 구간을 골라 2026 시간축에 붙이기

- 원본 LOT의 제외 뺀 행에서 `MINUTES × 60`개를 이어서 뽑음. 시작 위치는 고정된 난수로 고름
- 뽑은 행을 2026년 시작 시각부터 1초 간격으로 다시 놓음. 원본의 빈 초는 메우지 않고 행 순서만 씀
- 어느 원본 행에서 왔는지 `SOURCE_LOT`, `SOURCE_FROM`, `SOURCE_TO`로 남김

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

pieces = []
segment_log = []
for plan in SCHEDULE:
    rows = usable[usable["LOT_NO"] == plan["SOURCE_LOT"]].reset_index(drop=True)
    need = plan["MINUTES"] * 60

    if len(rows) < need:
        print(plan["LOT_NO"], "원본 행이 모자람:", len(rows), "<", need)
        continue

    start = int(rng.integers(0, len(rows) - need + 1))
    segment = rows.iloc[start:start + need].reset_index(drop=True)

    begin = pd.Timestamp(SIM_DATE + " " + plan["START"])
    piece = pd.DataFrame({
        "LOT_NO": plan["LOT_NO"],
        "MEAS_DT": begin + pd.to_timedelta(np.arange(need), unit="s"),
        "GRADE": plan["GRADE"],
        "SOURCE_MEAS_DT": segment["MEAS_DT"].values,  # 어느 원본 초에서 왔는지 (저장할 때는 뺌)
    })
    for channel in CHANNELS:
        piece[channel] = segment[channel].values
    pieces.append(piece)

    segment_log.append({
        "LOT_NO": plan["LOT_NO"], "GRADE": plan["GRADE"], "SOURCE_LOT": plan["SOURCE_LOT"],
        "SOURCE_FROM": segment["MEAS_DT"].iloc[0], "SOURCE_TO": segment["MEAS_DT"].iloc[-1],
        "SIM_FROM": piece["MEAS_DT"].iloc[0], "SIM_TO": piece["MEAS_DT"].iloc[-1], "ROWS": need,
    })

sim = pd.concat(pieces).reset_index(drop=True)
segment_log = pd.DataFrame(segment_log)
print("시뮬레이션 행:", len(sim))
segment_log

## 4. 흔들림 더하기

같은 값을 그대로 복사하면 2024 원본과 똑같은 행이 생김. 채널마다 그 LOT 구간 표준편차의 10% 크기로 정규 잡음을 더함. OP는 0~100 안으로 자르고, CP-Z4M은 값이 매우 작아 그대로 둠

In [ ]:
for plan in SCHEDULE:
    rows = sim["LOT_NO"] == plan["LOT_NO"]

    for channel in CHANNELS:
        if channel == "CP-Z4M":
            continue

        spread = sim.loc[rows, channel].std()
        if pd.isna(spread) or spread == 0:
            continue

        noise = rng.normal(0, spread * JITTER, size=int(rows.sum()))
        sim.loc[rows, channel] = sim.loc[rows, channel] + noise

        if channel.startswith("OP-"):
            sim.loc[rows, channel] = sim.loc[rows, channel].clip(0, 100)

print("흔들림 더함. 예:", CHANNELS[6], round(sim[CHANNELS[6]].std(), 3))

## 5. 옮겨 온 구간에 있던 팀 판단 1 표시하기

2024 구간을 통째로 옮기면 그 안에 팀이 이상 1로 판단한 사건(검토 표의 T 번호)도 함께 들어옴. 그 행은 `LABEL = 1`, `LABEL_REASON = 원본 T번호`로 표시함. 정상 0 판단은 따로 표시하지 않음

In [ ]:
DECISION_CSV = os.path.join(PDM_DIR, "data", "labels", "domain_decisions.csv")
decisions = pd.read_csv(DECISION_CSV, encoding=ENC)
decisions = decisions[(decisions["VERSION"] == "v6") & (decisions["DOMAIN_LABEL"] == 1)].copy()
decisions["START"] = pd.to_datetime(decisions["START"], errors="raise")
decisions["END"] = pd.to_datetime(decisions["END"], errors="raise")

sim["LABEL"] = 0
sim["LABEL_REASON"] = ""

carried = []
for plan in SCHEDULE:
    same_lot = decisions[decisions["LOT_NO"] == plan["SOURCE_LOT"]]

    for _, d in same_lot.iterrows():
        inside = (sim["LOT_NO"] == plan["LOT_NO"]) & (sim["SOURCE_MEAS_DT"] >= d["START"]) & (sim["SOURCE_MEAS_DT"] <= d["END"])
        count = int(inside.sum())
        if count == 0:
            continue

        sim.loc[inside, "LABEL"] = 1
        sim.loc[inside, "LABEL_REASON"] = "원본 " + d["DECISION_ID"]
        carried.append({"LOT_NO": plan["LOT_NO"], "DECISION_ID": d["DECISION_ID"], "SENSOR": d["SENSOR"],
                        "SIM_START": sim.loc[inside, "MEAS_DT"].min(), "SIM_END": sim.loc[inside, "MEAS_DT"].max(),
                        "SECONDS": count, "REASON": d["REASON"]})

carried = pd.DataFrame(carried)
print("옮겨 온 팀 판단 1:", len(carried), "건 /", int(sim["LABEL"].sum()), "행")
carried

## 6. 사건 심기

| 종류 | 방법 |
| --- | --- |
| 이식 | 2024 사건 구간의 CP-Z4에서 그 직전 10분 중앙값을 뺀 편차를 구하고, 2026 대상 구간의 CP-Z4에 그대로 더함. 온도와 출력은 그대로 둠 (CP만 튄 사건이라서) |
| 합성 | 램프 시간 동안 TC-Z4를 조금씩 내리고 OP-Z4를 조금씩 올림 (제어가 보상하려 하지만 온도가 못 따라오는 모양). 그 뒤 회복 시간 동안 되돌림 |

사건 행은 `LABEL = 1`로 표시하고 `LABEL_REASON`에 사건 번호를 적음

In [ ]:
event_log = []
for event in EVENTS:
    begin = pd.Timestamp(SIM_DATE + " " + event["START"])

    if event["KIND"] == "이식":
        src = source[(source["LOT_NO"] == event["SOURCE_LOT"]) &
                     (source["MEAS_DT"] >= event["SOURCE_START"]) &
                     (source["MEAS_DT"] <= event["SOURCE_END"])]
        before = source[(source["LOT_NO"] == event["SOURCE_LOT"]) &
                        (source["MEAS_DT"] < event["SOURCE_START"]) &
                        (source["MEAS_DT"] >= pd.Timestamp(event["SOURCE_START"]) - pd.Timedelta(minutes=10))]
        baseline = before["CP-Z4"].median()
        deviation = (src["CP-Z4"] - baseline).values

        end = begin + pd.Timedelta(seconds=len(deviation) - 1)
        target = (sim["LOT_NO"] == event["TARGET_LOT"]) & (sim["MEAS_DT"] >= begin) & (sim["MEAS_DT"] <= end)
        count = int(target.sum())

        sim.loc[target, "CP-Z4"] = sim.loc[target, "CP-Z4"].values + deviation[:count]
        detail = "원본 " + str(event["SOURCE_LOT"]) + " " + event["SOURCE_START"][5:16] + " ~ " + event["SOURCE_END"][11:16] + ", CP-Z4 편차 최대 " + str(round(float(np.abs(deviation).max()), 3))

    else:
        ramp = event["RAMP_MINUTES"] * 60
        recover = event["RECOVER_MINUTES"] * 60
        end = begin + pd.Timedelta(seconds=ramp + recover - 1)
        target = (sim["LOT_NO"] == event["TARGET_LOT"]) & (sim["MEAS_DT"] >= begin) & (sim["MEAS_DT"] <= end)
        count = int(target.sum())

        # 0에서 1까지 올라갔다가 다시 0으로 내려오는 진행 정도
        progress = np.concatenate([np.linspace(0, 1, ramp), np.linspace(1, 0, recover)])[:count]
        sim.loc[target, "TC-Z4"] = sim.loc[target, "TC-Z4"].values - event["TC_DROP"] * progress
        sim.loc[target, "OP-Z4"] = (sim.loc[target, "OP-Z4"].values + event["OP_RISE"] * progress).clip(0, 100)
        detail = "TC-Z4 -" + str(event["TC_DROP"]) + "℃, OP-Z4 +" + str(event["OP_RISE"]) + "%p, 램프 " + str(event["RAMP_MINUTES"]) + "분, 회복 " + str(event["RECOVER_MINUTES"]) + "분"

    sim.loc[target, "LABEL"] = 1
    sim.loc[target, "LABEL_REASON"] = event["EVENT_ID"]

    event_log.append({"EVENT_ID": event["EVENT_ID"], "KIND": event["KIND"], "TITLE": event["TITLE"],
                      "LOT_NO": event["TARGET_LOT"], "START": begin, "END": end, "SECONDS": count, "DETAIL": detail})

event_log = pd.DataFrame(event_log)
print("LABEL 1 행:", int(sim["LABEL"].sum()))
event_log

## 7. 저장

서버 등록 형식: `ROW_ID`, `LOT_NO`, `MEAS_DT`, `GRADE`, 22채널, `LABEL`, `EXCLUDED`, `LABEL_VERSION`, `LABEL_REASON`. `MEAS_DT`는 시간대 없는 `YYYY-MM-DD HH:MM:SS`

In [ ]:
output = sim.copy()
output.insert(0, "ROW_ID", ["sim-" + SIM_DATE + ":row:" + str(i + 1) for i in range(len(output))])
output["MEAS_DT"] = output["MEAS_DT"].dt.strftime("%Y-%m-%d %H:%M:%S")
output["EXCLUDED"] = False
output["LABEL_VERSION"] = LABEL_VERSION

save_columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE"] + CHANNELS + ["LABEL", "EXCLUDED", "LABEL_VERSION", "LABEL_REASON"]
output = output[save_columns]
output.to_csv(OUT_CSV, index=False, encoding=ENC)
event_log.to_csv(EVENT_CSV, index=False, encoding=ENC)

print("저장:", OUT_CSV)
print("행:", len(output), "/ LOT:", output["LOT_NO"].nunique(), "/ LABEL 1:", int(output["LABEL"].sum()), "/ 크기 MB:", round(os.path.getsize(OUT_CSV) / 1024 / 1024, 1))
output.groupby("LOT_NO").agg(강종=("GRADE", "first"), 시작=("MEAS_DT", "min"), 끝=("MEAS_DT", "max"), 행=("ROW_ID", "count"), 사건행=("LABEL", "sum"))

## 8. 확인 그림: 하루의 CP-Z4, TC-Z4, OP-Z4와 심은 사건

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 폰트 (팀 양식)
for name in ["Malgun Gothic", "NanumGothic", "AppleGothic"]:
    if any(f.name == name for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, channel in zip(axes, ["CP-Z4", "TC-Z4", "OP-Z4"]):
    ax.plot(sim["MEAS_DT"], sim[channel], linewidth=0.5, color="#2563EB")
    ax.set_ylabel(channel)
    ax.grid(alpha=0.3)

    for _, e in event_log.iterrows():
        ax.axvspan(e["START"], e["END"], color="#DC2626", alpha=0.15)
    for _, s in segment_log.iterrows():
        ax.axvline(s["SIM_FROM"], color="#6B7280", linestyle="--", linewidth=0.8)

for _, e in event_log.iterrows():
    axes[0].text(e["START"], axes[0].get_ylim()[1], e["EVENT_ID"], color="#DC2626", va="top", fontsize=9)
for _, s in segment_log.iterrows():
    axes[0].text(s["SIM_FROM"], axes[0].get_ylim()[0], " " + s["LOT_NO"] + " " + s["GRADE"], color="#6B7280", va="bottom", fontsize=8)

axes[0].set_title(SIM_DATE + " 시뮬레이션 하루: 점선은 LOT 경계, 빨간 띠는 심은 사건")
axes[2].set_xlabel("시각")
plt.tight_layout()
plt.savefig(FIG_PNG, dpi=120)
plt.show()